###Delta Lake (Lakehouse Performance Optimization, Cost Saving & Best Practices)

###Performance Optimization

In [0]:
%sql
use retail.schema_retail;

In [0]:
%sql
CREATE OR REPLACE TABLE tblsales
(
  sales_id INT,
  product_id INT,
  region STRING,
  sales_amount DOUBLE,
  sales_date DATE
)
USING DELTA;

In [0]:
%sql
describe history tblsales;


In [0]:
%sql
describe detail tblsales;

In [0]:
%sql
select * from tblsales;

In [0]:
%sql
INSERT INTO tblsales VALUES
  (1, 101, 'North', 1000.50, '2025-10-16'),
  (2, 102, 'South', 500.75, '2025-10-16'),
  (3, 103, 'East', 700.20, '2025-10-16'),
  (4, 104, 'West', 1200.00, '2025-10-16');

INSERT INTO tblsales VALUES
  (5, 101, 'North', 800.00, '2025-10-17'),
  (6, 102, 'South', 450.00, '2025-10-17'),
  (7, 103, 'East', 600.00, '2025-10-17'),
  (8, 104, 'West', 1100.00, '2025-10-17');


In [0]:
%sql
select * from tblsales;

In [0]:
%sql
--Check fragmentation (numFiles & sizeInBytes)
DESCRIBE DETAIL tblsales;

In [0]:
%sql
--Optimize the table
--This performs file compaction:
--Combines many small Parquet files into fewer large files (around 1 GB default).
--Improves read performance and reduces metadata overhead.
OPTIMIZE tblsales;

In [0]:
%sql
DESCRIBE DETAIL tblsales;

#### 2. ZORDER
- ZORDER is an optional feature used with OPTIMIZE to colocate related data physically in the same set of files by sorting and add internal indexing for faster retrival of data.
- Reduces file scan for queries filtering on ZORDER columns using index.
- Works best for columns (low or high cardinal) used frequently in WHERE clauses.

#### EXAMPLE USE CASE:
- Periodically optimize large Delta tables with frequent writes/updates (optimize will compact)
- Use ZORDER on high-selectivity/filtering columns to improve read performance (ordering and indexing happens behind the scene).


In [0]:
%sql

-- Step 1 – Create the Delta table
use retail.schema_retail;
CREATE OR REPLACE TABLE customer_txn (
    txn_id INT,
    customer_id INT,
    region STRING,
    txn_amount DOUBLE,
    txn_type STRING,
    transaction_date DATE
)
USING DELTA;

In [0]:
%sql
describe history customer_txn

In [0]:
%sql
--Step 2 – Insert multiple small batches
--Each insert writes a few small Parquet files.
-- Batch 1
INSERT INTO customer_txn VALUES
 (1, 1001, 'North', 250.00, 'Online', '2025-10-01'),
 (2, 1002, 'South', 400.00, 'Offline', '2025-10-02'),
 (3, 1003, 'West', 600.00, 'Online', '2025-10-03');

-- Batch 2
INSERT INTO customer_txn VALUES
 (4, 1001, 'North', 300.00, 'Offline', '2025-10-01'),
 (5, 1004, 'East', 750.00, 'Online', '2025-10-02'),
 (6, 1005, 'South', 180.00, 'Online', '2025-10-03');

-- Batch 3
INSERT INTO customer_txn VALUES
 (7, 1001, 'North', 270.00, 'Online', '2025-10-01'),
 (8, 1003, 'West', 500.00, 'Offline', '2025-10-02'),
 (9, 1002, 'South', 900.00, 'Online', '2025-10-03');

/*
--Before optimize or zordering
tablefolder
    - part-0
    - part-1
    - part-2
    - part-3
    - part-4
    - part-5

select * from customer_txn where region='North; --This query will scan all files and give the result of matching data

optimize customer_txn;
After optimize
tablefolder
    - part-0
    - part-1
    - part-2
    - part-3
    - part-4
    - part-5 -- All part-0 to part-4 will be kept together in part-5

select * from customer_txn where region='North; --This query will scan part-5 file alone and give the result of matching data (reduced files operation & metadata management)


optimize customer_txn zorder by transaction_date;

optimize customer_txn;
After optimize
tablefolder
    - part-0
    - part-1
    - part-2
    - part-3
    - part-4
    - part-5 -- All part-0 to part-4 will be kept together in part-5 + sort + colocate + indexing the data rows on transaction_date column

select * from customer_txn where transaction_date='2025-12-10'; --This query will scan part-5 file alone using the index and scan only the data wrt 2025-12-10 alone without querying the entire data and give the result (reduced files operation + metadata management + faster retrival of required data alone without making full table scan)
*/





In [0]:
%sql
describe history customer_txn

In [0]:
%sql
-- Step 3 – Inspect fragmentation (numFiles & sizeInBytes)
DESCRIBE DETAIL customer_txn;

In [0]:
%sql
-- Step 4 – Run OPTIMIZE ZORDER - watch out the metrics - zOrderStats
-- Now compact and physically order data with index added.
OPTIMIZE customer_txn ZORDER BY (transaction_date);

In [0]:
%sql
--look at the operationParameters (zOrderBy)
DESCRIBE HISTORY customer_txn

In [0]:
%sql
-- Step 3 – Inspect fragmentation
DESCRIBE DETAIL customer_txn;

####3. Partitioning
Partitioning is the practice of physically splitting a table's data into separate **folders** based on a column.<br>
Good partition columns:<br>
- Low cardinality (low difference columns such as date, age, city, region, gender)
- Columns used Frequently used in filters
- Hard to manage (we can't change the partition columns very frequently)